# Atari PPO with the package Nature CNN extractor

This notebook uses the same Atari preprocessing stack as the native REINFORCE tutorial, but trains `stable_baselines3.PPO` with `NatureCNNExtractor` supplied through `policy_kwargs`.
            

In [ ]:
# Uncomment in a fresh environment:
# %pip install "crosslearn[atari]"

In [ ]:
import gymnasium as gym

try:
    import ale_py
    print(f"ale_py successfully imported from: {ale_py.__file__}")
except ImportError:
    print("ale_py is not found or not importable.")
    
atari_env_ids = sorted(
    env_id for env_id in gym.registry.keys() if str(env_id).startswith('ALE/')
)
print(f'Registered ALE environments: {len(atari_env_ids)}')
print('Sample ids:', atari_env_ids[:12])

ENV_ID = 'ALE/Breakout-v5'
if ENV_ID not in atari_env_ids:
    raise RuntimeError(
        f'{ENV_ID} is not registered. Install the Atari extra before running this notebook.'
    )

## Preprocessing and extractor wiring

The environment logic stays the same as the native notebook. The only SB3-specific part is the PPO constructor and its `policy_kwargs`. For PPO tuning details, use the SB3 documentation as the primary reference.
            

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

from crosslearn.envs import AtariPreprocessor
from crosslearn.extractors import NatureCNNExtractor


def make_env():
    env = gym.make(ENV_ID, render_mode='rgb_array', frameskip=1)
    return AtariPreprocessor(env, stack_size=4, frame_skip=1, screen_size=84)

sample_env = make_env()
print('Observation space:', sample_env.observation_space)
print('Action space:', sample_env.action_space)
sample_env.close()        

In [ ]:
env = make_env()
model = PPO(
    'CnnPolicy',
    env,
    policy_kwargs={
        'features_extractor_class': NatureCNNExtractor,
        'features_extractor_kwargs': {'features_dim': 512},
    },
    verbose=1,
    seed=42,
)
model.learn(total_timesteps=100_000)

In [ ]:
eval_env = make_env()
obs, info = eval_env.reset(seed=42)
terminated = False
truncated = False
episode_return = 0.0

while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(int(action))
    episode_return += reward

print(f'Evaluation return on {ENV_ID}: {episode_return:.2f}')
eval_env.close()      